# T9: Map visualization of T2 data quality results

Uses T2 quality checks applied per taxi zone, and T3 trip volume per zone.

In [ ]:
%%sql -r wh
USE WAREHOUSE BIGDATA_MZMB_WH

In [ ]:
import pandas as pd
import numpy as np
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

quality = session.sql('SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.T9_QUALITY_BY_ZONE').to_pandas()
volume = session.sql('SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.T3_PICKUP_LOCATION_DIST').to_pandas()
geom = session.sql('SELECT LOCATION_ID, BOROUGH, ZONE, ST_ASGEOJSON(GEOM) AS GEOJSON FROM BIGDATA_TAXI_MZMB.EXTERNAL_DATA.TAXI_ZONES_GEOM').to_pandas()

print(f"Quality (T2 checks per zone): {len(quality)} rows, {quality['DATASET'].nunique()} datasets")
print(f"Volume (T3): {len(volume)} rows")
print(f"Zone geometries: {len(geom)} zones")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
import matplotlib.colors as mcolors

def parse_geojson_polygons(geojson_str):
    g = json.loads(geojson_str)
    polys = []
    if g['type'] == 'Polygon':
        polys.append(g['coordinates'][0])
    elif g['type'] == 'MultiPolygon':
        for poly in g['coordinates']:
            polys.append(poly[0])
    return polys

zone_polys = {}
for _, row in geom.iterrows():
    try:
        polys = parse_geojson_polygons(row['GEOJSON'])
        zone_polys[int(row['LOCATION_ID'])] = polys
    except:
        pass

print(f"Parsed polygons for {len(zone_polys)} zones")

In [ ]:
def draw_zone_map(ax, data_df, colormap, norm, id_col='LOCATION_ID', val_col='TOTAL_ISSUE_PCT'):
    data_df = data_df.copy()
    data_df[id_col] = data_df[id_col].astype(int)
    for _, row in data_df.iterrows():
        loc_id = row[id_col]
        if loc_id not in zone_polys:
            continue
        val = min(row[val_col], norm.vmax)
        color = colormap(norm(val))
        for coords in zone_polys[loc_id]:
            poly = MplPolygon(coords, closed=True, facecolor=color, edgecolor='black', linewidth=0.2, alpha=0.8)
            ax.add_patch(poly)
    for loc_id, polys in zone_polys.items():
        if loc_id not in data_df[id_col].values:
            for coords in polys:
                poly = MplPolygon(coords, closed=True, facecolor='#f0f0f0', edgecolor='gray', linewidth=0.15, alpha=0.4)
                ax.add_patch(poly)
    ax.set_xlim(-74.27, -73.7)
    ax.set_ylim(40.49, 40.92)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

datasets = ['yellow', 'green', 'fhv', 'fhvhv']
titles = ['Yellow taxi', 'Green taxi', 'FHV', 'FHVHV (Uber/Lyft)']

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
vmax = quality['TOTAL_ISSUE_PCT'].quantile(0.95)
norm = mcolors.Normalize(vmin=0, vmax=vmax)
cmap = plt.colormaps['YlOrRd']

for ax, ds, title in zip(axes.flatten(), datasets, titles):
    subset = quality[quality['DATASET'] == ds]
    draw_zone_map(ax, subset, cmap, norm)
    ax.set_title(title, fontsize=11)

fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Data quality issues (% of trips)', fontsize=10)
plt.suptitle('T2 data quality issues by taxi zone', fontsize=13, y=0.95)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
vmax = volume['TRIP_COUNT'].quantile(0.95)
norm = mcolors.LogNorm(vmin=max(volume['TRIP_COUNT'].min(), 1), vmax=vmax)
cmap = plt.colormaps['Blues']

for ax, ds, title in zip(axes.flatten(), datasets, titles):
    subset = volume[volume['DATASET'] == ds].rename(columns={'PU_LOCATION_ID': 'LOCATION_ID'})
    draw_zone_map(ax, subset, cmap, norm, val_col='TRIP_COUNT')
    ax.set_title(title, fontsize=11)

fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Trip count (log scale)', fontsize=10)
plt.suptitle('Trip volume by taxi zone (from T3)', fontsize=13, y=0.95)
plt.show()